In [1]:
pip install sentence-transformers faiss-cpu numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 39.2 MB/s eta 0:00:00


In [2]:
import os
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
DB_DIR = "dense_index_db"
INDEX_PATH = os.path.join(DB_DIR, "faiss_index.bin")
METADATA_PATH = os.path.join(DB_DIR, "paragraph_metadata.json")

os.makedirs(DB_DIR, exist_ok=True)

# Define standard embedding size for sentence-transformers/all-MiniLM-L6-v2
EMBEDDING_DIM = 384

print("🤖 Loading sentence-transformers/all-MiniLM-L6-v2 model...")
# Automatically downloads and caches the model on first fresh run
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# ==========================================
# 2. CREATE DEV-SET PARAGRAPH POOL
# ==========================================
# Mock Dev Corpus representing a document dataset needing compilation into a pool
raw_documents = {
    "doc_1": """Dense vector retrieval leverages neural network encoders to map texts into a continuous vector space. This allows downstream applications to perform semantic matches instead of exact string token matches. FAISS is perfectly suited for managing these nearest neighbor workloads efficiently.""",
    "doc_2": """FAISS (Facebook AI Similarity Search) is an optimized C++ library with elegant Python wrappers designed specifically for ultra-fast dense vector clustering and geometric similarity calculations. It balances memory allocation constraints against extreme processing speeds.""",
    "doc_3": """When preprocessing paragraph pools for production development sets, always ensure clean line breaks are standard. Strip stray markdown artifacts or excessive white space from raw text strings to prevent corrupt tokenizing behavior inside small transformers like MiniLM."""
}

paragraph_pool = []
paragraph_metadata = []

print("📝 Processing raw text into paragraph pools...")
for doc_id, text in raw_documents.items():
    # Split text by paragraph barriers (using double linebreaks or custom chunks as logic commands)
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]

    for idx, para in enumerate(paragraphs):
        paragraph_pool.append(para)
        paragraph_metadata.append({
            "global_id": len(paragraph_pool) - 1,
            "source_doc": doc_id,
            "local_chunk_index": idx,
            "text": para
        })

print(f"✅ Generated a dev pool containing {len(paragraph_pool)} unique paragraphs.")

# ==========================================
# 3. COMPUTE EMBEDDINGS & BUILD FAISS INDEX
# ==========================================
print("⚡ Generating dense embeddings with MiniLM-L6-v2...")
# Extract embeddings (Returns raw NumPy arrays)
embeddings = model.encode(paragraph_pool, show_progress_bar=True, convert_to_numpy=True)

# Normalize vectors for Cosine Similarity (Inner Product Indexing)
faiss.normalize_L2(embeddings)

print("🏗️ Initializing FAISS Index Flat IP (Inner Product)...")
# IndexFlatIP handles cosine similarity calculations directly on normalized vectors
index = faiss.IndexFlatIP(EMBEDDING_DIM)

# Register the text embeddings to the geometric matrix
index.add(embeddings)
print(f"📊 Vector footprint currently mapped in FAISS: {index.ntotal}")

# ==========================================
# 4. SAVE AND UTILITY EXPORT
# ==========================================
print("💾 Persisting FAISS Index and mapping configuration metadata to disk...")
faiss.write_index(index, INDEX_PATH)

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(paragraph_metadata, f, ensure_ascii=False, indent=4)

print(f"✨ Build Complete! Files stored safely inside current workspace at: '{DB_DIR}/'\n")

# ==========================================
# 5. DRY-RUN RE-LOAD & QUERY VALIDATION
# ==========================================
print("🧪 Executing isolated query test against newly built index...")
# Read files straight from directory to ensure fresh clone works immediately
loaded_index = faiss.read_index(INDEX_PATH)
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    loaded_meta = json.load(f)

test_query = "How fast can FAISS perform geometric searches?"
query_vector = model.encode([test_query], convert_to_numpy=True)
faiss.normalize_L2(query_vector)

# Query top K matches
K = 1
scores, indices = loaded_index.search(query_vector, K)

print("\n--- Search Query Results Summary ---")
print(f"Query: \"{test_query}\"")
for rank in range(K):
    meta_idx = indices[0][rank]
    print(f"Top Hit Score: {scores[0][rank]:.4f}")
    print(f"Source Document Context: {loaded_meta[meta_idx]['source_doc']}")
    print(f"Extracted String: \"{loaded_meta[meta_idx]['text']}\"\n")


🤖 Loading sentence-transformers/all-MiniLM-L6-v2 model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

📝 Processing raw text into paragraph pools...
✅ Generated a dev pool containing 3 unique paragraphs.
⚡ Generating dense embeddings with MiniLM-L6-v2...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🏗️ Initializing FAISS Index Flat IP (Inner Product)...
📊 Vector footprint currently mapped in FAISS: 3
💾 Persisting FAISS Index and mapping configuration metadata to disk...
✨ Build Complete! Files stored safely inside current workspace at: 'dense_index_db/'

🧪 Executing isolated query test against newly built index...

--- Search Query Results Summary ---
Query: "How fast can FAISS perform geometric searches?"
Top Hit Score: 0.5252
Source Document Context: doc_2
Extracted String: "FAISS (Facebook AI Similarity Search) is an optimized C++ library with elegant Python wrappers designed specifically for ultra-fast dense vector clustering and geometric similarity calculations. It balances memory allocation constraints against extreme processing speeds."



In [7]:
%%writefile build_pipeline.py

Writing build_pipeline.py


In [11]:
!python build_pipeline.py


🤖 Loading model...
Loading weights: 100% 103/103 [00:00<00:00, 9513.83it/s]
⚡ Embedding...

🔍 Query: 'Is Facebook library quick at vector calculations?'
🎯 Match: 'FAISS (Facebook AI Similarity Search) is an optimized library designed for fast similarity calculations.' (Score: 0.5154)


In [ ]:
#Test code inside the build_pipeline.py
import os
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# 1. SETUP
DB_DIR = "dense_index_db"
INDEX_PATH = os.path.join(DB_DIR, "faiss_index.bin")
METADATA_PATH = os.path.join(DB_DIR, "paragraph_metadata.json")
os.makedirs(DB_DIR, exist_ok=True)

print("🤖 Loading model...")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 2. MOCK DATA
raw_documents = {
    "doc_1": "Dense vector retrieval leverages neural network encoders to map texts into a continuous vector space.",
    "doc_2": "FAISS (Facebook AI Similarity Search) is an optimized library designed for fast similarity calculations.",
}

paragraph_pool = []
paragraph_metadata = []

for doc_id, text in raw_documents.items():
    paragraph_pool.append(text)
    paragraph_metadata.append({"global_id": len(paragraph_pool) - 1, "source_doc": doc_id, "text": text})

# 3. EMBED & INDEX
print("⚡ Embedding...")
embeddings = model.encode(paragraph_pool, convert_to_numpy=True)
faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(384)
index.add(embeddings)

# 4. SAVE
faiss.write_index(index, INDEX_PATH)
with open(METADATA_PATH, "w") as f:
    json.dump(paragraph_metadata, f, indent=4)

# 5. TEST QUERY
test_query = "Is Facebook library quick at vector calculations?"
query_vector = model.encode([test_query], convert_to_numpy=True)
faiss.normalize_L2(query_vector)

scores, indices = index.search(query_vector, 1)
print(f"\n🔍 Query: '{test_query}'")
print(f"🎯 Match: '{paragraph_metadata[indices[0][0]]['text']}' (Score: {scores[0][0]:.4f})")
